# ESC (Electronic Speed Controller)

> **What an ESC does:** It sits between your battery and motor, taking a low-power control signal (typically PWM from a flight controller or receiver) and converting battery voltage into precisely timed power pulses that drive a brushless motor. Essentially it's the translator between "rotate at 60% throttle" and the actual electrical switching needed to make that happen.---

- skip_showdoc: true
- skip_exec: true

### How it works internally

An ESC contains a few key subsystems:

**Power stage (MOSFETs):** Six MOSFETs arranged in three half-bridge pairs — one high-side and one low-side per motor phase. By turning these on and off in a precise sequence, the ESC synthesises the three-phase alternating current a brushless motor needs. The MOSFET quality largely determines efficiency and maximum current.

**Microcontroller (MCU):** Runs the firmware (BLHeli_S, BLHeli_32, AM32, etc.) and handles commutation timing, current sensing, temperature monitoring, and communication with the flight controller. Higher-end ESCs use 32-bit ARM Cortex-M MCUs (hence "BLHeli_32") versus older 8-bit AVR chips.

**Commutation:** There are two approaches — *trapezoidal* (6-step, simpler, slightly coarser) and *sinusoidal/FOC* (Field-Oriented Control, smoother torque, more efficient, better at low speeds). FOC is increasingly common in high-quality FPV and robotics ESCs.---

### Key specifications

**Continuous current rating (amps):** The most important spec. This is the sustained current the ESC can handle without overheating — e.g. 30A, 45A, 65A. Always choose a rating above your motor's max draw, with some headroom (20–30% is common advice).

**Voltage / cell count (S rating):** LiPo packs are rated in cells — 3S (11.1V), 4S (14.8V), 6S (22.2V). Your ESC must support the voltage of your battery. Higher voltage = same power at lower current = less heat in wires, but motors and props must match.

**Burst current:** The short-duration (a few seconds) current the ESC can handle above its continuous rating. Useful for FPV freestyle where throttle punches are brief.

**BEC (Battery Eliminator Circuit):** Some ESCs include a built-in voltage regulator (BEC) to power the flight controller/receiver directly from the main battery — eliminates a separate regulator. Others are "opto" (no BEC, signal isolated optically) — you'll need an external BEC.

---

### Firmware & protocols

| Firmware | MCU | Notes |
|---|---|---|
| BLHeli_S | 8-bit (EFM8) | Mature, widely used, supports DSHOT |
| BLHeli_32 | 32-bit ARM | Smoother, telemetry, FOC capable |
| AM32 | 32-bit ARM | Open source BLHeli_32 alternative |
| SimonK | 8-bit AVR | Old multirotors, largely superseded |
| VESC | 32-bit ARM | Open source, robotics/e-boards, full FOC |

**Signal protocols** (what the flight controller sends):

- **PWM** — legacy 50Hz signal, 1000–2000µs pulse width. Slow update rate.
- **OneShot125 / OneShot42** — faster PWM variants, 125µs / 42µs pulse.
- **Multishot** — up to 32kHz, very low latency.
- **DSHOT** (300/600/1200) — fully digital, no calibration needed, supports telemetry back-channel, immune to electrical noise. The current standard for FPV.
- **PWM analog (for servos)** — some ESCs for cars/boats still use standard servo PWM.

---

### Form factors

**4-in-1 ESC:** All four ESCs on a single PCB, designed to stack directly with the flight controller via standoffs. Cleaner build, less weight, easier wiring — the default for modern FPV quads.

**Individual ESC:** One per motor, soldered to a power distribution board or directly to the battery leads. Easier to replace one failed unit, common on larger or heavier machines.

**Integrated FC+ESC (AIO):** Flight controller and four ESCs on one board. Ultra-compact — popular in sub-250g and whoop-class builds.

---

### Protection features to look for

- **Overcurrent / short circuit protection** — cuts power if current spikes dangerously
- **Over-temperature protection** — throttles or cuts when the ESC gets too hot
- **Low-voltage cutoff (LVC)** — protects LiPo cells from over-discharge (more important for fixed-wing/cars than FPV quads where you use OSD for voltage)
- **Active braking / regenerative braking** — relevant for cars, boats, and e-bikes; in FPV, "active freewheeling" affects motor braking on prop wash
- **Desync protection** — BLHeli_32/AM32 handle motor sync loss more gracefully than older firmware

---

### FPV-specific tips (since you run quads)

- **30×30 and 20×20 stacks** are the standard mounting patterns for 4-in-1 ESCs
- **DSHOT600** is the sweet spot for most builds — fast enough, well-supported, very reliable
- **Bidirectional DSHOT + RPM filtering** (available in BLHeli_32/AM32 + Betaflight) feeds motor RPM data back to the FC for notch filtering — dramatically cleaner gyro signal
- Heat is the primary killer — make sure there's airflow over the ESC in your frame, or add a heatsink pad
- For 5" freestyle on 4S, a 45A 4-in-1 is a comfortable spec; 6S builds generally want 40A+ even at lower power since voltage spikes stress the FETs more